## Notebook to start a spark instance in an ArcGIS Pro Session.

### THE BELOW HAS TO BE THE FIRST IMPORT !

In [1]:
from spark_esri import spark_start, spark_stop

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [3]:
import os

import arcpy
import geofunctions as S

### Check if the jar exists

In [4]:
spark_jars = os.path.expanduser("~/geofunctions-0.10.jar")
os.path.exists(spark_jars)

True

In [5]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    "spark.jars": spark_jars,
}
spark = spark_start(config=config)

In [6]:
spark.version

'3.5.1-esri-1'

### Register the geo functions with SparkSQL.

In [22]:
S.st_register_functions()

### Sample usage. Create random points in the current map extent and place them on the Map.

In [7]:
project = arcpy.mp.ArcGISProject("CURRENT")

extent = project.activeView.camera.getExtent()
xmin = extent.XMin
ymin = extent.YMin
dx = extent.XMax - extent.XMin
dy = extent.YMax - extent.YMin

In [8]:
(
    spark
    # Create 1000 points.
    .range(1000)
    # Get random x values
    .withColumn("x", F.rand() * dx + xmin)
    # Get random y values.
    .withColumn("y", F.rand() * dy + ymin)
    # Create a point geometry from implicit x/y.
    .withColumn("geom", S.st_point())
    # Convert the spark dataframe to a feature class to be displayed in the TOC.
    .to_feature_class(
        "MyPoints",
        sp_ref = -1,  # Use current map SR.
    )
)

### Create a spark dataframe from a feature class.

In [15]:
df = S.to_spark("MyPoints")

In [16]:
df.printSchema()

root
 |-- OBJECTID: long (nullable = true)
 |-- Shape: binary (nullable = true)



In [13]:
df.count()

1000

In [21]:
(df
     .select(
        "OBJECTID",
        S.st_astext("Shape"),
    )
     .where(F.col("OBJECTID").between(10,15))
     .show(truncate=False)
)

+--------+--------------------------------------------+
|OBJECTID|stastext(Shape)                             |
+--------+--------------------------------------------+
|10      |POINT (-8269207.7939 4957789.162199996)     |
|11      |POINT (-8269001.9213 4944507.054700002)     |
|12      |POINT (-8259964.025 4961879.095200002)      |
|13      |POINT (-8245396.327299999 4938301.1206)     |
|14      |POINT (-8263351.0184 4950723.1032000035)    |
|15      |POINT (-8262341.657500001 4937090.707999997)|
+--------+--------------------------------------------+



### Create a Spark SQL table from the dataframe and apply some SQL functions.

In [23]:
df.createOrReplaceTempView("v")

In [25]:
spark.sql("""
select OBJECTID,ST_ASTEXT(Shape) as WKT
from v
where OBJECTID between 10 and 15
""").show(truncate=False)

+--------+--------------------------------------------+
|OBJECTID|WKT                                         |
+--------+--------------------------------------------+
|10      |POINT (-8269207.7939 4957789.162199996)     |
|11      |POINT (-8269001.9213 4944507.054700002)     |
|12      |POINT (-8259964.025 4961879.095200002)      |
|13      |POINT (-8245396.327299999 4938301.1206)     |
|14      |POINT (-8263351.0184 4950723.1032000035)    |
|15      |POINT (-8262341.657500001 4937090.707999997)|
+--------+--------------------------------------------+

